In [8]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
from sklearn.model_selection import cross_val_score

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report,roc_auc_score
import joblib

In [9]:
df = pd.read_csv("../data/feature_engineered_data.csv")

---->Train and testing model<----

In [10]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]
numerical_cols = [
    "person_age",
    "person_income",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "credit_score"
]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

final_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42
)

final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

## Evaluation metrics :

In [11]:
print("Accuracy score :",accuracy_score(y_test,y_pred))

print("\nPrecision score :",precision_score(y_test,y_pred))

print("\nRecall score :",recall_score(y_test,y_pred))

print("\nF1 score :",f1_score(y_test,y_pred))
print("\n")

report = classification_report(
    y_test,
    y_pred
)

print(report)


# roc - auc

y_prob = final_model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test,y_prob)

print(auc)

# feature importance 
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": final_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

Accuracy score : 0.9316590732303589

Precision score : 0.8959405374499714

Recall score : 0.7835

F1 score : 0.8359562550013336


              precision    recall  f1-score   support

           0       0.94      0.97      0.96      6999
           1       0.90      0.78      0.84      2000

    accuracy                           0.93      8999
   macro avg       0.92      0.88      0.90      8999
weighted avg       0.93      0.93      0.93      8999

0.9756104443491928
                           Feature  Importance
7   previous_loan_defaults_on_file    0.226167
4              loan_percent_income    0.166546
3                    loan_int_rate    0.163011
1                    person_income    0.134766
2                        loan_amnt    0.064140
6                     credit_score    0.061474
12      person_home_ownership_RENT    0.057138
0                       person_age    0.036611
5       cb_person_cred_hist_length    0.029253
14     loan_intent_HOMEIMPROVEMENT    0.010094
11     

In [12]:
joblib.dump(final_model, "../models/random_forest.pkl")

joblib.dump(scaler, "../models/scaler.pkl")

feature_columns = X_train.columns.tolist()

joblib.dump(feature_columns, "../models/feature_columns.pkl")

print(feature_columns)

['person_age', 'person_income', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file', 'person_education_Bachelor', 'person_education_Doctorate', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']
